# 01 · What a tensor is

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/01-what-a-tensor-is.ipynb)

*Part I · demo · 20 min*

> 🇪🇸 **Qué es un tensor** — Aprende a leer la estructura de un tensor y a seguir el significado de sus ejes mientras los fijas, reorganizas o contraes.

Learn to read a tensor's structure and track what its axes mean as you fix, rearrange, or contract them.

## What you will be able to do

- Explain **order**, **axis/mode**, and **shape**, and distinguish tensor order from matrix/tensor rank.
- Read `.shape`, `.ndim`, and `.size` and explain what every axis means on real image data.
- Predict how **slices**, **fibers**, **unfolding**, and **contraction** change or preserve axes.
- Use `np.einsum` for a dot product and matrix multiplication and reason about the output shape.

> 🇪🇸 **Al terminar podrás:**
> - Explicar **orden**, **eje/modo** y **forma**, y distinguir el orden tensorial del rango matricial/tensorial.
> - Leer `.shape`, `.ndim` y `.size` e interpretar qué significa cada eje en datos reales de imágenes.
> - Predecir cómo **cortes**, **fibras**, **unfolding** y **contracción** cambian o conservan los ejes.
> - Usar `np.einsum` para un producto escalar y una multiplicación matricial, razonando sobre la forma de salida.

## Setup

Run this first. It installs and imports everything this notebook needs, and nothing else.

> 🇪🇸 Ejecuta esto primero: instala e importa todo lo que este cuaderno necesita.

## Why this matters

For the data tensors used in this workshop, shape alone is not enough; we also need to know what each axis represents. Two arrays can both be order 3 while their axes describe completely different things. Tracking which axes are fixed, kept, rearranged, or summed will make every later tensor operation easier to reason about.

**Learning cycle:** **Predict → Run → Explain.** Try to predict shapes and axis meanings before you execute each example.

> 🇪🇸 **Por qué esto importa:** Para los tensores de datos usados en este taller, la forma por sí sola no es suficiente; también necesitamos saber qué representa cada eje. Dos arreglos pueden ser ambos de orden 3 y, aun así, sus ejes representar conceptos completamente diferentes. Seguir qué ejes se fijan, conservan, reorganizan o suman facilita el razonamiento sobre las operaciones tensoriales posteriores.
>
> **Ciclo de aprendizaje:** **Predice → Ejecuta → Explica.** Intenta anticipar las formas y el significado de los ejes antes de ejecutar cada ejemplo.

In [ ]:
import numpy as np
from sklearn.datasets import load_digits
from skimage import data

rng = np.random.default_rng(0)

## 1.1 Vocabulary

Keep this table open for the whole workshop. Do not memorize it all at once—use it while you predict what each operation does.

> 🇪🇸 Mantén esta tabla abierta durante el taller. No necesitas memorizarla de una vez: úsala mientras predices qué hace cada operación.

| Term | Plain meaning | Spanish | Example |
|---|---|---|---|
| **Tensor** | An array of numbers with any number of axes | *tensor* | A colour image |
| **Axis** (pl. axes) | One direction along which data is arranged | *eje* | Height; width; colour |
| **Mode** | Another word for axis, used in tensor theory | *modo* | "mode-0 unfolding" |
| **Order** | How many axes a tensor has | *orden* | A matrix has order 2 |
| **Shape** | The size along each axis, as a tuple | *forma* | `(512, 512, 3)` |
| **Slice** | Fix one index, keep the rest | *corte* | One colour channel |
| **Fiber** | Fix every index except one | *fibra* | The 3 colour values of one pixel |
| **Unfolding** | Rearrange a tensor into a matrix while preserving all entries | *desplegado / unfolding* | Mode-2 image unfolding |
| **Contraction** | Multiply and sum over selected/shared indices | *contracción* | Dot product |
| **Decomposition** | Represent an array using simpler structured components | *descomposición* | SVD, Tucker, CP |

### Order vs. Rank

In this workshop, **order** refers to the number of axes/modes a tensor has. **Matrix rank** measures linear independence. **Tensor rank** has its own definitions and is not the same as tensor order.

> 🇪🇸 **Orden vs. rango:** El **orden** se refiere al número de ejes/modos de un tensor. El **rango matricial** mide la independencia lineal. El **rango tensorial** tiene sus propias definiciones y no es lo mismo que el orden tensorial.

**Quick check:** if an array has shape `(32, 8, 8)`, what is its order? Can you infer its matrix/tensor rank from the shape alone?

> 🇪🇸 **Comprobación rápida:** si un arreglo tiene forma `(32, 8, 8)`, ¿cuál es su orden? ¿Puedes inferir su rango matricial/tensorial solo a partir de la forma?

<details>
<summary><strong>Check your reasoning · Comprueba tu razonamiento</strong></summary>

Its **order is 3** because it has three axes. Its matrix/tensor rank **cannot be inferred from the shape alone**.

> 🇪🇸 Su **orden es 3** porque tiene tres ejes. Su rango matricial/tensorial **no se puede inferir solo a partir de la forma**.

</details>

## 1.2 Shape in NumPy

Every NumPy array has `.shape`, a tuple giving the size along each axis. The length of that tuple is `.ndim`, the number of axes, and `.size` is the total number of stored values.

> 🇪🇸 **Forma en NumPy:** Todo arreglo de NumPy tiene `.shape`, una tupla con el tamaño de cada eje. La longitud de esa tupla es `.ndim`, el número de ejes, y `.size` es el número total de valores almacenados.

In [ ]:
scalar = np.array(3.0)                     # book: a           — order 0
vector = np.array([1., 2., 3.])            # book: x, x_i      — order 1
matrix = np.array([[1., 2.], [3., 4.]])    # book: A, A_{i,j}  — order 2
tensor = rng.standard_normal((2, 3, 4))    # book: A_{i,j,k}   — order 3

for name, arr in [("scalar", scalar), ("vector", vector),
                  ("matrix", matrix), ("tensor", tensor)]:
    print(f"{name:8s} shape={str(arr.shape):12s} ndim={arr.ndim}  size={arr.size}")

These tiny synthetic arrays are deliberate: they isolate structure — order, shape, `ndim`, and `size` — without distracting domain details. We switch immediately afterward to real image data to reason about what each axis means.

> 🇪🇸 **Por qué usamos datos sintéticos aquí:** Estos arreglos pequeños permiten aislar la estructura — orden, forma, `ndim` y `size` — sin detalles del dominio. Inmediatamente después usamos imágenes reales para razonar sobre el significado de cada eje.

A scalar has `shape=()`, an empty tuple—there are no axes to measure. `size` is the product of the dimensions in `shape`: for `(2, 3, 4)`, `size = 2 × 3 × 4 = 24`.

Now move from deliberately simple synthetic arrays to **real data**, where axis meaning matters.

> 🇪🇸 Un escalar tiene `shape=()`, una tupla vacía: no hay ejes que medir. `size` es el producto de las dimensiones de `shape`: para `(2, 3, 4)`, `size = 2 × 3 × 4 = 24`.
>
> Ahora pasamos de arreglos sintéticos simples a **datos reales**, donde el significado de cada eje sí importa.

### Predict before running

What do you expect the shapes to be for `digits.images` and `photo`? What do their axes represent?

> 🇪🇸 **Predice antes de ejecutar:** `digits.images` y `photo` son ambos tensores de orden 3. Antes de ejecutar, predice qué representa cada uno de sus tres ejes. ¿Significan lo mismo?

In [ ]:
digits = load_digits()
print(digits.images.shape)      # (1797, 8, 8)  — 1797 handwritten digits, 8x8 pixels

photo = data.immunohistochemistry()
print(photo.shape)              # (512, 512, 3) — height, width, colour

Both arrays are order 3, but their axes mean completely different things. `digits.images` counts **images** along axis 0; `photo` counts **colour channels** along axis 2. **Shape describes structure, not semantics.** You must know what every axis represents and keep track of that meaning.

> 🇪🇸 Ambos arreglos son de orden 3, pero sus ejes significan cosas completamente diferentes. `digits.images` cuenta **imágenes** en el eje 0; `photo` cuenta **canales de color** en el eje 2. **La forma describe estructura, no semántica.** Debes saber qué representa cada eje y seguir ese significado durante las operaciones.

## Exercise 1 — read the shapes

**Predict first.** Build the arrays, then verify `.shape`, `.ndim`, and `.size`. For the real image tensors, explain what every axis counts.

> 🇪🇸 **Predice primero.** Construye los arreglos y después verifica `.shape`, `.ndim` y `.size`. Para los tensores de imágenes reales, explica qué cuenta cada eje.

In [ ]:
# TODO 1: Build a scalar, a vector, a matrix and an order-3 tensor, and print
#         .shape, .ndim and .size for each. Which one has shape ()?

# TODO 2: Take load_digits().images and data.astronaut(). Both are order 3.
#         For each, write down in a comment what axis 0, 1 and 2 count.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
for arr in [np.array(3.0), np.zeros(3), np.zeros((2, 2)), np.zeros((2, 3, 4))]:
    print(arr.shape, arr.ndim, arr.size)
# ()        0 1
# (3,)      1 3
# (2, 2)    2 4
# (2, 3, 4) 3 24

print(load_digits().images.shape)   # (1797, 8, 8)   axis 0 = which image
                                    #                axis 1 = row of pixels
                                    #                axis 2 = column of pixels
print(data.astronaut().shape)       # (512, 512, 3)  axis 0 = height
                                    #                axis 1 = width
                                    #                axis 2 = colour channel

<details>
<summary><b>Why this solution works · Por qué funciona esta solución</b></summary>

1.  A scalar has no axes, so its shape is `()`. A vector has one axis, a matrix two, and an order-3 tensor three. The `.ndim` attribute directly tells you the number of axes (order), and `.size` is the total number of elements.
2.  `load_digits().images` represents a collection of 8x8 pixel images. So, axis 0 counts the images, axis 1 counts the rows of pixels, and axis 2 counts the columns of pixels. `data.astronaut()` is a color image. Axis 0 counts height, axis 1 counts width, and axis 2 counts the color channels (Red, Green, Blue).

> 🇪🇸 **Por qué funciona esta solución:**
>
> 1.  Un escalar no tiene ejes, por lo que su forma es `()`. Un vector tiene un eje, una matriz dos y un tensor de orden 3 tres. El atributo `.ndim` indica directamente el número de ejes (orden), y `.size` es el número total de elementos.
> 2.  `load_digits().images` representa una colección de imágenes de 8x8 píxeles. Por lo tanto, el eje 0 cuenta las imágenes, el eje 1 cuenta las filas de píxeles y el eje 2 cuenta las columnas de píxeles. `data.astronaut()` es una imagen en color. El eje 0 cuenta la altura, el eje 1 la anchura y el eje 2 los canales de color (Rojo, Verde, Azul).
</details>

## 1.3 The three operations that matter

Slices/fibers, unfolding, and contraction all answer one question: **what happens to the axes?**

> 🇪🇸 Cortes/fibras, unfolding y contracción responden a una misma pregunta: **¿qué ocurre con los ejes?**

### Slices and fibers — fixing indices takes a tensor apart

A **slice** fixes one index and keeps the others. A **fiber** fixes every index except one.

> 🇪🇸 Un **corte** fija un índice y conserva los demás. Una **fibra** fija todos los índices excepto uno.

In [ ]:
print(photo[:, :, 0].shape)       # (512, 512) — a slice: one colour channel, still an image
print(photo[100, 200, :].shape)   # (3,)       — a fiber: the 3 colour values of one pixel

Same picture, same two indexing operations — see them together. Drag the
sliders and watch the marked pixel move on both panels at once, while its
fiber (three numbers, one per colour) redraws on the right.

> 🇪🇸 Mueve los deslizadores: el mismo píxel se marca en el corte y en la
> imagen completa, y su fibra (tres números, uno por color) se redibuja.

In [ ]:
# Colab renders ipywidgets through its own widget manager rather than the
# classic Jupyter one; this call is a no-op outside Colab, which is why it is
# guarded rather than assumed.
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass

import ipywidgets as widgets
import matplotlib.pyplot as plt

def show_slice_and_fiber(row, col):
    plt.close('all')
    fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))

    axes[0].imshow(photo)
    axes[0].scatter([col], [row], color='#C44E52', s=70, edgecolor='white')
    axes[0].set_title('photo — the fiber, marked')
    axes[0].axis('off')

    axes[1].imshow(photo[:, :, 0], cmap='gray')
    axes[1].scatter([col], [row], color='#C44E52', s=70, edgecolor='white')
    axes[1].set_title('photo[:, :, 0] — a slice')
    axes[1].axis('off')

    fiber = photo[row, col, :]
    axes[2].bar(['R', 'G', 'B'], fiber, color=['#C44E52', '#55A868', '#4C72B0'])
    axes[2].set_title(f'photo[{row}, {col}, :] — the fiber')
    axes[2].set_ylim(0, 255)

    plt.tight_layout()
    plt.show()

widgets.interact(show_slice_and_fiber,
                  row=widgets.IntSlider(min=0, max=511, step=1, value=100, description='row'),
                  col=widgets.IntSlider(min=0, max=511, step=1, value=200, description='col'));

### Unfolding — rearranging axes into a matrix

Mode unfoldings are central to many tensor methods, including the Tucker/HOSVD route used later in this workshop. An unfolding moves one axis to the front and rearranges the remaining axes into a matrix without losing entries.

> 🇪🇸 **Desplegado:** Los unfoldings por modo son fundamentales en muchos métodos tensoriales, incluida la ruta Tucker/HOSVD que usaremos más adelante. El desplegado reorganiza las entradas en una matriz sin perder información.

In [ ]:
def unfold(T, axis):
    """Move `axis` to the front, flatten everything else into one long axis."""
    return np.moveaxis(T, axis, 0).reshape(T.shape[axis], -1)

print(unfold(photo, 0).shape)   # (512, 1536) — rows are the height axis
print(unfold(photo, 2).shape)   # (3, 262144) — rows are the 3 colour channels

Unfolding **loses nothing**: it only rearranges entries. The mode-2 unfolding says “each colour channel is one row of 262,144 numbers”, which makes matrix tools such as SVD available without discarding information.

You will use this same idea again in sections 07 and 10.

> 🇪🇸 El unfolding **no pierde información**: solo reorganiza las entradas. En el unfolding de modo 2, cada canal de color se convierte en una fila de 262.144 números, lo que permite aplicar herramientas matriciales como SVD sin descartar datos.

### Contraction — multiply along a shared axis and sum over it

The dot product (eq. 2.8) and the matrix product (eq. 2.5) are both contractions. `np.einsum` makes the summed and retained indices explicit.

> 🇪🇸 **Contracción:** el producto escalar y el producto matricial son contracciones. `np.einsum` permite ver explícitamente qué índices se suman y cuáles permanecen.

### Predict before running

For `np.einsum('i,i->', a, b)`:
- which index is summed?
- which indices remain?
- why is the result a scalar?

For `np.einsum('ik,kj->ij', A, B)`:
- which index is summed?
- which indices remain?
- what should the output shape be?

> 🇪🇸 **Predice antes de ejecutar:**
>
> Para `np.einsum('i,i->', a, b)`:
> - ¿qué índice se suma?
> - ¿qué índices quedan?
> - ¿por qué el resultado es un escalar?
>
> Para `np.einsum('ik,kj->ij', A, B)`:
> - ¿qué índice se suma?
> - ¿qué índices quedan?
> - ¿cuál debería ser la forma de salida?

In [ ]:
a = np.array([1., 2., 3.]); b = np.array([4., 5., 6.])
print(np.einsum('i,i->', a, b))          # dot product, sum over i          (eq 2.8)

A = np.array([[1., 2.], [3., 4.]]); B = np.array([[5., 6.], [7., 8.]])
print(np.einsum('ik,kj->ij', A, B))      # matrix product, sum over k       (eq 2.5)

**The rule, in one sentence:** an index that appears in the inputs but **not** after the arrow is summed over; an index that appears after the arrow is kept.

This rule is the core idea that section 06 develops.

> 🇪🇸 **La regla, en una frase:** un índice que aparece en las entradas pero **no** después de la flecha se suma; un índice que aparece después de la flecha se conserva.
>
> Esta regla es la idea central que desarrolla la sección 06.

## Exercise 2 — take a tensor apart and put it back

Before coding, say out loud which axes you expect to **fix**, **keep**, **rearrange**, or **sum**. Then verify your reasoning with NumPy.

> 🇪🇸 Antes de programar, explica qué ejes esperas **fijar**, **conservar**, **reorganizar** o **sumar**. Después verifica tu razonamiento con NumPy.

In [ ]:
# TODO 3: From `photo`, extract (a) the green channel as a (512, 512) slice and
#         (b) the colour fiber at pixel (10, 20). Which is a slice, which a fiber?

# TODO 4: Unfold `photo` along all three axes and print the three shapes.
#         Confirm that each unfolding has exactly photo.size entries —
#         unfolding rearranges, it never loses anything.

# TODO 5: Write the dot product of `a` and `b` as einsum, and check it against
#         np.dot. Then write the matrix product of A and B, and check against @.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
green = photo[:, :, 1]        # slice — one index fixed, the rest kept
fiber = photo[10, 20, :]      # fiber — every index fixed except one
print(green.shape, fiber.shape)                # (512, 512) (3,)

for ax in range(3):
    M = unfold(photo, ax)
    print(ax, M.shape, M.size == photo.size)   # True every time

print(np.einsum('i,i->', a, b), np.dot(a, b))              # 32.0 32.0
print(np.allclose(np.einsum('ik,kj->ij', A, B), A @ B))    # True

<details>
<summary><b>Why this solution works · Por qué funciona esta solución</b></summary>

1.  `photo[:, :, 1]` selects the green channel by fixing the last axis (color) to index 1. This is a **slice** because one index is fixed and the rest are kept. `photo[10, 20, :]` selects the color values for the pixel at row 10, column 20. This is a **fiber** because all indices except one are fixed.
2.  `unfold(photo, axis)` rearranges the tensor. For `axis=0`, it creates a matrix where rows correspond to the height dimension. For `axis=1`, rows correspond to the width dimension. For `axis=2`, rows correspond to the color dimension. In all cases, the total number of elements (`.size`) remains the same, demonstrating that unfolding is merely a rearrangement.
3.  `np.einsum('i,i->', a, b)` performs the dot product by summing over the shared index `i`. `np.einsum('ik,kj->ij', A, B)` performs matrix multiplication by summing over the shared index `k` and keeping `i` and `j`. `np.allclose` confirms the results are numerically equivalent to `np.dot` and `@` operator respectively.

> 🇪🇸 **Por qué funciona esta solución:**
>
> 1.  `photo[:, :, 1]` selecciona el canal verde fijando el último eje (color) al índice 1. Esto es un **corte** porque un índice se fija y el resto se mantienen. `photo[10, 20, :]` selecciona los valores de color para el píxel en la fila 10, columna 20. Esto es una **fibra** porque todos los índices excepto uno están fijos.
> 2.  `unfold(photo, axis)` reorganiza el tensor. Para `axis=0`, crea una matriz donde las filas corresponden a la dimensión de altura. Para `axis=1`, las filas corresponden a la dimensión de anchura. Para `axis=2`, las filas corresponden a la dimensión de color. En todos los casos, el número total de elementos (`.size`) permanece igual, demostrando que el desplegado es solo una reorganización.
> 3.  `np.einsum('i,i->', a, b)` realiza el producto escalar sumando sobre el índice compartido `i`. `np.einsum('ik,kj->ij', A, B)` realiza la multiplicación de matrices sumando sobre el índice compartido `k` y manteniendo `i` y `j`. `np.allclose` confirma que los resultados son numéricamente equivalentes a `np.dot` y al operador `@` respectivamente.
</details>

## Exercise 3 — Axis Reasoning Challenge

Let `D = load_digits().images`, with shape `(1797, 8, 8)` = **images × height × width**.

**Predict before running:** for each expression below, write the expected output shape and explain which axes are fixed, kept, rearranged, or contracted:

1. `D[0]`
2. `D[:, 3, 4]`
3. `unfold(D, 0)`
4. `np.einsum('nhw->n', D)`

Then run your code and compare your prediction with the result.

> 🇪🇸 **Reto de razonamiento sobre ejes:** Sea `D = load_digits().images`, con forma `(1797, 8, 8)` = **imágenes × alto × ancho**.
>
> **Predice antes de ejecutar:** para cada expresión, escribe la forma de salida esperada y explica qué ejes se fijan, conservan, reorganizan o contraen. Después ejecuta el código y compara tu predicción con el resultado.

In [ ]:
# TODO 6: Let D = load_digits().images.
#         Before running each operation, predict its output shape.
#
#         1. D[0]
#         2. D[:, 3, 4]
#         3. unfold(D, 0)
#         4. np.einsum('nhw->n', D)
#
#         For each operation, explain in a comment which axes were
#         fixed, kept, rearranged, or summed/contracted.

In [ ]:
#@title Solution — try it yourself first { display-mode: 'form' }
D = load_digits().images
print("1.", D[0].shape)
print("2.", D[:, 3, 4].shape)
print("3.", unfold(D, 0).shape)
print("4.", np.einsum('nhw->n', D).shape)

<details>
<summary><b>Why this solution works · Por qué funciona esta solución</b></summary>

The operations change the shape as follows:

1.  `D[0]` takes the first image. The `n` axis is fixed, `h` and `w` are kept. Resulting shape: `(8, 8)`. This is a **slice**.
2.  `D[:, 3, 4]` takes the pixels at row 3, column 4 from all images. The `h` and `w` axes are fixed, `n` is kept. Resulting shape: `(1797,)`. This is a **fiber**.
3.  `unfold(D, 0)` rearranges the tensor. The `n` axis is kept as the first dimension, and `h` and `w` are flattened. Resulting shape: `(1797, 64)`. This is an **unfolding**.
4.  `np.einsum('nhw->n', D)` sums over `h` and `w` axes. The `n` axis is kept. Resulting shape: `(1797,)`. This is a **contraction**.

> 🇪🇸 **Por qué funciona esta solución:** Las operaciones cambian la forma de la siguiente manera:
>
> 1.  `D[0]` toma la primera imagen. El eje `n` se fija, `h` y `w` se mantienen. Forma resultante: `(8, 8)`. Esto es un **corte**.
> 2.  `D[:, 3, 4]` toma los píxeles en la fila 3, columna 4 de todas las imágenes. Los ejes `h` y `w` se fijan, `n` se mantiene. Forma resultante: `(1797,)`. Esto es una **fibra**.
> 3.  `unfold(D, 0)` reorganiza el tensor. El eje `n` se mantiene como la primera dimensión, y `h` y `w` se aplanan. Forma resultante: `(1797, 64)`. Esto es un **desplegado**.
> 4.  `np.einsum('nhw->n', D)` suma sobre los ejes `h` y `w`. El eje `n` se mantiene. Forma resultante: `(1797,)`. Esto es una **contracción**.
</details>

## 1.4 The map of factorizations (Preview)

This is only a preview—**do not memorize these methods yet**. Later sections move from familiar matrix factorizations such as SVD to tensor decompositions such as Tucker.

- **Tucker** represents a tensor using a smaller core tensor and factor matrices.
- **CP** represents a tensor as a sum of rank-one components.

These methods build on the tensor vocabulary developed here; the Tucker/HOSVD route used later explicitly uses mode unfoldings.

> 🇪🇸 **Vista previa:** Esto es solo un adelanto—**todavía no necesitas memorizar estos métodos**. Más adelante pasaremos de factorizaciones matriciales como SVD a descomposiciones tensoriales como Tucker.
>
> - **Tucker** representa un tensor mediante un tensor núcleo más pequeño y matrices de factores.
> - **CP** representa un tensor como una suma de componentes de rango uno.
>
> La ruta Tucker/HOSVD que se usa más adelante emplea explícitamente unfoldings por modo.

## What just happened

You should now be able to reason about a tensor by following its axes:

- `shape`, `ndim`, and `size` describe **structure**;
- axis labels describe **meaning**;
- slices/fibers **fix indices**;
- unfolding **rearranges entries without losing them**;
- contraction **sums selected axes**.

**One final self-check:** if you cannot explain what each output axis represents, go back one step and trace the indices again.

> 🇪🇸 **Qué acaba de suceder:** Ahora deberías poder razonar sobre un tensor siguiendo sus ejes:
>
> - `shape`, `ndim` y `size` describen la **estructura**;
> - las etiquetas de los ejes describen el **significado**;
> - los cortes/fibras **fijan índices**;
> - el unfolding **reorganiza entradas sin perderlas**;
> - la contracción **suma ejes seleccionados**.
>
> **Autoevaluación final:** si no puedes explicar qué representa cada eje de salida, vuelve un paso atrás y sigue de nuevo los índices.

---

## Done with this section

Next up: **02 · Thinking in N dimensions** — [open in Colab](https://colab.research.google.com/github/project-delphi/tensors-workshop/blob/main/notebooks/02-thinking-in-n-dimensions.ipynb).

[← Back to the workshop site](https://project-delphi.github.io/tensors-workshop/) · [All notebooks](https://project-delphi.github.io/tensors-workshop/notebooks.html) · [Handbook](https://project-delphi.github.io/tensors-workshop/tensors_workshop_plan_with_quizzes.html)